In [ ]:
# =============================================================================
# Knowledge Graph Extraction — Fine-Tuning Notebook
# Model  : Meta-Llama-3.1-8B-Instruct (4-bit via Unsloth)
# GPU    : Kaggle 2x T4 (16 GB each) — Unsloth uses GPU 0 only; GPU 1 is idle
# Dataset: Jotschi/wikipedia_knowledge_graph_en (17M rows, streaming subset)
# Task   : Given text → structured JSON {nodes, edges}
#
# ⚠️ Before running:
#   1. Settings → Accelerator → GPU T4 x2
#   2. Settings → Internet → ON  (needed for HuggingFace downloads)
#   3. No Kaggle dataset attachments needed — everything loads from HuggingFace
# =============================================================================

## Cell 1 — Install Dependencies
Run once; restart runtime not required on Kaggle.

In [1]:
import subprocess, sys

packages = [
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "trl>=0.8.6",
    "datasets>=2.19",
    "xformers",
    "bitsandbytes",
    "huggingface_hub",
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

print("✅ Installation done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 16.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 31.3 MB/s eta 0:00:00
✅ Installation done


## Cell 2 — Configuration
All tunable knobs in one place.

In [1]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME        = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
# Lighter alternative if you hit OOM or want faster epochs:
# MODEL_NAME      = "unsloth/Phi-3.5-mini-instruct-bnb-4bit"  # 3.8B, ~half the memory

MAX_SEQ_LENGTH    = 512   # Trim long Wikipedia passages to fit T4 VRAM

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_RANK         = 8     # 8 → faster/smaller; 32 → more capacity

# ── Dataset ────────────────────────────────────────────────────────────────
HF_DATASET        = "Jotschi/wikipedia_knowledge_graph_en"
NUM_TRAIN_SAMPLES = 25_000  # Subset of the 17M rows; increase for better quality
                             # 25k → ~1.5 h training on T4; 50k → ~3 h

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE        = 1
GRAD_ACCUM        = 8       # Effective batch = 2 × 4 = 8
MAX_STEPS         = 600     # ~600 steps covers 25k samples at eff. batch 8 once
WARMUP_RATIO      = 0.05
LR                = 2e-4
SEED              = 42

# ── Paths ──────────────────────────────────────────────────────────────────
OUTPUT_DIR        = "/kaggle/working/kg_extractor"
LORA_SAVE_PATH    = f"{OUTPUT_DIR}/lora_adapters"
GGUF_SAVE_PATH    = f"{OUTPUT_DIR}/gguf_model"

print("✅ Config loaded")

✅ Config loaded


## Cell 3 — Load Base Model (4-bit, Unsloth)

In [2]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,          # auto: bfloat16 on Ampere+, float16 on T4
    load_in_4bit  = True,
)

# Pad token fix (Llama-3 uses EOS as pad by default)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

gpu = torch.cuda.get_device_properties(0)
print(f"✅ Model loaded on {gpu.name}  |  VRAM: {gpu.total_memory/1e9:.1f} GB")
print(f"   Reserved so far: {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.3: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


✅ Model loaded on Tesla T4  |  VRAM: 15.6 GB
   Reserved so far: 5.78 GB


## Cell 4 — Inject LoRA Adapters

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_RANK,
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
    lora_alpha          = LORA_RANK,     # alpha == rank → scale = 1
    lora_dropout        = 0,             # 0 is optimal for Unsloth
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # saves ~30% VRAM
    random_state        = SEED,
    use_rslora          = False,
)
model.print_trainable_parameters()
print("✅ LoRA applied")

Unsloth 2026.6.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
✅ LoRA applied


## Cell 5 — Load & Inspect the Dataset
We stream a shuffled subset so we never download all 17 M rows.

In [ ]:
from datasets import load_dataset

# streaming=True → iterate lazily; we materialise only NUM_TRAIN_SAMPLES rows
raw = load_dataset(HF_DATASET, split="train", streaming=True)
raw = raw.shuffle(buffer_size=10_000, seed=SEED)
raw = raw.take(NUM_TRAIN_SAMPLES)

# Materialise to a regular Dataset so we can .map() and .filter() efficiently
from datasets import Dataset
raw = Dataset.from_generator(lambda: (x for x in raw))

print(f"✅ Loaded {len(raw):,} samples")
print(f"   Columns: {raw.column_names}")
print(f"\n── Sample row ──────────────────────────────────────────────────────")
import json, pprint
pprint.pprint(raw[0])

Generating train split: 0 examples [00:00, ? examples/s]

## Cell 6 — Dataset Formatting
#
⚠️  Check the **sample row** printed above and set `TEXT_FIELD` and
    `GRAPH_FIELD` to the correct column names for your dataset.

In [ ]:
# ── Adjust these two lines after inspecting Cell 5 output ─────────────────
TEXT_FIELD  = "text"           # column containing the source passage
GRAPH_FIELD = "ontology"       # column containing the extracted graph (JSON / dict)
# Common alternatives: "graph", "output", "relations", "triples"
# ──────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are a knowledge graph extraction expert. "
    "Given a text passage, extract all named entities and relationships "
    "and return ONLY a JSON object matching this exact schema:\n"
    '{"nodes": [{"id": "<entity_name>", "type": "<entity_type>"}], '
    '"edges": [{"source": "<entity1>", "target": "<entity2>", '
    '"relation": "<RELATION_TYPE>"}]}\n'
    "Return valid JSON only. No explanation, no markdown."
)


def format_example(example):
    """Convert one dataset row into a chat-formatted training string."""
    text = str(example.get(TEXT_FIELD, "") or "").strip()

    # Parse graph ground-truth
    raw_graph = example.get(GRAPH_FIELD, {})
    if isinstance(raw_graph, str):
        try:
            raw_graph = json.loads(raw_graph)
        except json.JSONDecodeError:
            raw_graph = {}

    # Normalise to {"nodes": [...], "edges": [...]} if needed.
    # Some datasets use "entities"/"relations" or "triples" — adapt here.
    if isinstance(raw_graph, dict):
        nodes = raw_graph.get("nodes", raw_graph.get("entities", []))
        edges = raw_graph.get("edges", raw_graph.get("relations",
                              raw_graph.get("relationships", [])))
        graph_json = json.dumps({"nodes": nodes, "edges": edges},
                                ensure_ascii=False)
    else:
        graph_json = json.dumps(raw_graph, ensure_ascii=False)

    # Truncate long passages so they fit inside MAX_SEQ_LENGTH
    text = text[:2000]

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Extract a knowledge graph from:\n\n{text}"},
        {"role": "assistant", "content": graph_json},
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }


formatted = raw.map(
    format_example,
    num_proc=4,
    remove_columns=raw.column_names,
    desc="Formatting",
)

# Drop anything that came out empty or clearly broken
formatted = formatted.filter(lambda x: len(x["text"]) > 200)

print(f"✅ Formatted dataset: {len(formatted):,} rows")
print("\n── Formatted sample (first 600 chars) ──────────────────────────────")
print(formatted[0]["text"][:600])

## Cell 7 — Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model               = model,
    tokenizer           = tokenizer,
    train_dataset       = formatted,
    dataset_text_field  = "text",
    max_seq_length      = MAX_SEQ_LENGTH,
    dataset_num_proc    = 2,
    packing             = True,          # packs short sequences → more tokens/step
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio                = WARMUP_RATIO,
        max_steps                   = MAX_STEPS,
        learning_rate               = LR,
        fp16      = not torch.cuda.is_bf16_supported(),
        bf16      =     torch.cuda.is_bf16_supported(),
        logging_steps               = 25,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = SEED,
        output_dir                  = OUTPUT_DIR,
        save_strategy               = "steps",
        save_steps                  = 150,
        save_total_limit            = 2,
        report_to                   = "none",   # set "wandb" if you want tracking
    ),
)

print("── Pre-training VRAM ───────────────────────────────────────────────")
print(f"   Reserved: {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

trainer_stats = trainer.train()

print("\n── Training complete ───────────────────────────────────────────────")
print(f"   Peak VRAM : {torch.cuda.max_memory_reserved()/1e9:.2f} GB")
print(f"   Steps     : {trainer_stats.global_step}")
print(f"   Train loss: {trainer_stats.training_loss:.4f}")

## Cell 8 — Quick Inference Test (sanity check)

In [ ]:
FastLanguageModel.for_inference(model)

test_texts = [
    "Google was founded by Larry Page and Sergey Brin at Stanford University in 1998.",
    "Python is a programming language created by Guido van Rossum and first released in 1991.",
    "The Eiffel Tower was designed by Gustave Eiffel and built for the 1889 World's Fair in Paris.",
]

for txt in test_texts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Extract a knowledge graph from:\n\n{txt}"},
    ]
    ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to("cuda")

    with torch.inference_mode():
        out = model.generate(
            ids,
            max_new_tokens = 512,
            temperature    = 1e-6,   # near-zero → greedy
            do_sample      = False,
        )

    result = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
    print(f"\nInput : {txt}")
    print(f"Output: {result}")
    print("─" * 70)

## Cell 9 — Save LoRA Adapters
These are lightweight (~100 MB) — the base model is NOT stored again.
You can push them to HuggingFace Hub or keep them locally.

In [ ]:
import os
os.makedirs(LORA_SAVE_PATH, exist_ok=True)

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"✅ LoRA adapters saved → {LORA_SAVE_PATH}")

# Optional: push to your HuggingFace Hub repo
# model.push_to_hub("your-hf-username/kg-llama3-lora")
# tokenizer.push_to_hub("your-hf-username/kg-llama3-lora")

## Cell 10 — Export to GGUF (for Ollama / llama.cpp)
This merges LoRA weights into the base model and converts to 4-bit GGUF.
Takes ~10–15 min on T4. The resulting file is ~5 GB — download it from
the Kaggle output panel on the right.
#
Skip this cell if you only need the LoRA adapters (e.g. for HF transformers).

In [ ]:
os.makedirs(GGUF_SAVE_PATH, exist_ok=True)

model.save_pretrained_gguf(
    GGUF_SAVE_PATH,
    tokenizer,
    quantization_method = "q4_k_m",   # good quality/size balance
)

print(f"✅ GGUF model saved → {GGUF_SAVE_PATH}")
print("\nTo use in Ollama:")
print(f"  1. Download the .gguf file from the Kaggle output panel")
print( "  2. Create a file named 'Modelfile' with:")
print(f"       FROM ./<filename>.gguf")
print( "       PARAMETER temperature 0")
print( "  3. Run: ollama create kg-extractor -f Modelfile")
print( "  4. In your pipeline: OLLAMA_MODEL = 'kg-extractor'")